# 01 · 感測器 EDA（預測性維護）

探索 `toy_sensors.csv`（5 台機台 × 150 時點）：溫度 / 振動 / 電流與 `failure` 的關係。
教學重點：先看懂資料分布與「特徵 → 標籤」直覺，再進特徵工程。

> 下一格會自動把 capstone 根目錄加進 `sys.path`，所以從任何目錄啟動 Jupyter 都能 `from src...` import。

In [ ]:
# ── 路徑啟動：讓本 notebook 不論從哪個目錄開啟都能 import src ──
# notebook 沒有 __file__，且 Jupyter 的工作目錄就是 notebook 所在的 notebooks/，
# capstone 根目錄不在 sys.path 上，直接 import src 會 ModuleNotFoundError。
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "src" / "data" / "loaders.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        print("capstone root =", _base)
        break
else:
    raise FileNotFoundError("找不到 capstone 根目錄（應含 src/data/loaders.py）")

In [ ]:
from src.data.loaders import load_sensors
from src.data.validation import validate_sensors

# 載入 + 驗證（缺真實資料時自動後援 toy_sensors）。
df = load_sensors()
validate_sensors(df)
print(df.shape)
df.head()

In [ ]:
# 各機台基本統計 + 故障率，快速建立資料直覺。
summary = df.groupby("machine_id").agg(
    n=("failure", "size"),
    failure_rate=("failure", "mean"),
    temp_mean=("temperature", "mean"),
    vib_mean=("vibration", "mean"),
)
summary

In [ ]:
# 對比「故障 vs 正常」時的感測器分布，驗證『溫度高+振動大→故障』假設。
df.groupby("failure")[["temperature", "vibration", "current"]].mean()